<a href="https://colab.research.google.com/github/brevfeed/pyspark/blob/main/notebooks/10-partitioning-shuffle/01-repartition-coalesce-by-column.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

*Part of the [BrevFeed PySpark notebooks](https://github.com/brevfeed/pyspark). Runs on real Spark 4.0 — for the in-browser course see [brevfeed.com/learn/pyspark](https://brevfeed.com/learn/pyspark).*


In [ ]:
# --- Colab setup: run me first (no-op if you already have Spark) --------------
# Installs a JDK + PySpark and fetches the sample data. ~1-2 min on a cold
# Colab runtime; instant on re-run. Safe to run locally too.
import os
import re
import subprocess
import sys
import urllib.request

DATA_URL = "https://raw.githubusercontent.com/brevfeed/pyspark/main/data"
DATA_DIR = "data"
IN_COLAB = "google.colab" in sys.modules


def _java_version():
    """Major version of the JDK on PATH, or 0 if there isn't one."""
    try:
        out = subprocess.run(["java", "-version"], capture_output=True, text=True).stderr
    except (FileNotFoundError, OSError):
        return 0
    m = re.search(r'version "(\d+)', out)
    return int(m.group(1)) if m else 0


# Spark 4 requires Java 17+. Colab ships an older JDK on some images.
if _java_version() < 17:
    if IN_COLAB:
        print("Installing OpenJDK 17 ...")
        subprocess.run(["apt-get", "-qq", "update"], check=False)
        subprocess.run(
            ["apt-get", "-qq", "install", "-y", "openjdk-17-jdk-headless"], check=True
        )
        os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
    else:
        raise SystemExit(
            "Java 17+ is required for Spark 4. Install a JDK 17 and set JAVA_HOME."
        )

try:
    import pyspark  # noqa: F401
except ImportError:
    print("Installing PySpark ...")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "pyspark==4.0.0"], check=True
    )

# Sample data lives in the repo; pull only what's missing.
os.makedirs(DATA_DIR, exist_ok=True)
for _f in ("orders.csv", "customers.csv", "events.jsonl", "sample_logs.txt"):
    _p = os.path.join(DATA_DIR, _f)
    if not os.path.exists(_p):
        urllib.request.urlretrieve(f"{DATA_URL}/{_f}", _p)


def spark_ui(spark=None):
    """Open the Spark UI — and make it reachable from Colab.

    The UI is served by the driver on the *runtime*, so on Colab plain
    http://localhost:4040 is not something your browser can reach. Colab can
    proxy a runtime port, so ask it for a real URL. Call this AFTER the
    SparkSession exists, or there is no server on the port yet.
    """
    port = 4040
    if spark is not None:  # the driver bumps the port if 4040 was taken
        try:
            port = int(spark.sparkContext.uiWebUrl.rsplit(":", 1)[1])
        except Exception:
            pass
    if not IN_COLAB:
        return f"http://localhost:{port}"
    from google.colab.output import eval_js

    url = eval_js(f"google.colab.kernel.proxyPort({port})")
    print(f"Spark UI: {url}  (open in a new tab)")
    return url


print("Ready. Sample data in ./data:", sorted(os.listdir(DATA_DIR)))
print("Spark UI: call spark_ui(spark) after creating the session.")


# 10.1 — repartition, coalesce, and repartition-by-column

X-ray the four distribution rules in `explain()`, watch where rows
actually land with `spark_partition_id()`, time the coalesce
stage-fusion trap, and fix a small-files write for real.

In [ ]:
import os, sys
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import col

spark = (
    SparkSession.builder
    .appName("10.1-repartition-coalesce")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.sql.adaptive.enabled", "false")   # static plans first; AQE returns at the end
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

orders = spark.read.csv(f"{DATA_DIR}/orders.csv",
                        header=True, inferSchema=True)
print("orders partitions on read:", orders.rdd.getNumPartitions())

## The four plan signatures

One `explain()` each. Find the node named in the lesson's table —
and note which one has no `Exchange` at all.

In [ ]:
orders.repartition(16).explain()                        # RoundRobinPartitioning
orders.repartition("country").explain()                 # hashpartitioning, n = 8 (shuffle.partitions!)
orders.repartitionByRange(4, "order_date").explain()    # rangepartitioning
orders.coalesce(2).explain()                            # Coalesce — no Exchange

## Where the rows actually land

`spark_partition_id()` is the per-row x-ray. Round-robin comes out
flat; hash comes out lumpy with empties (co-location, not balance);
range comes out banded AND balanced.

In [ ]:
def where_did_rows_land(df, label):
    print(f"--- {label}")
    (df.groupBy(F.spark_partition_id().alias("pid"))
       .agg(F.count("*").alias("rows"),
            F.collect_set("country").alias("countries"))
       .orderBy("pid")
       .show(truncate=False))

where_did_rows_land(orders.repartition(4), "round-robin: repartition(4)")
where_did_rows_land(orders.repartition(8, "country"),
                    "hash: repartition(8, country) — empties + lumps")
where_did_rows_land(orders.repartitionByRange(4, "order_date"),
                    "range: repartitionByRange(4, order_date) — bands")

## The coalesce fusion trap, timed

Identical expensive computation, then "make it one partition" two
ways. `coalesce(1)` has no Exchange, so the whole upstream stage
runs at parallelism 1; `repartition(1)` pays one shuffle to keep
the computation wide. Check the Jobs tab: the coalesce run's single
stage has ONE task.

In [ ]:
import time

def expensive():
    df = spark.range(3_000_000).withColumn(
        "h", F.sha2(col("id").cast("string"), 512))
    for _ in range(4):                      # pile on CPU work
        df = df.withColumn("h", F.sha2(col("h"), 512))
    return df

for label, narrow in [("coalesce(1)", expensive().coalesce(1)),
                      ("repartition(1)", expensive().repartition(1))]:
    start = time.perf_counter()
    narrow.write.format("noop").mode("overwrite").save()
    print(f"{label:>15}: {time.perf_counter() - start:5.1f} s")

## Fixing a small-files write

Each task writes its own file into each partitionBy directory it
holds rows for. We fake a shuffled upstream with `repartition(8)`,
then fix it by co-locating each country into one task first.

In [ ]:
def file_report(path):
    total = 0
    for root, _, files in sorted(os.walk(path)):
        parqs = [f for f in files if f.endswith(".parquet")]
        if parqs:
            total += len(parqs)
            print(f"  {os.path.basename(root):>12}: {len(parqs)} file(s)")
    print(f"  {'TOTAL':>12}: {total}")

shuffled_upstream = orders.repartition(8)   # simulates post-shuffle state

shuffled_upstream.write.mode("overwrite") \
    .partitionBy("country").parquet("output/naive")
print("naive — every task writes into every directory it touches:")
file_report("output/naive")

shuffled_upstream.repartition("country").write.mode("overwrite") \
    .partitionBy("country").parquet("output/tidy")
print("\nrepartition(country) first — one task owns each country:")
file_report("output/tidy")

## The modern default: REBALANCE (Spark 3.2+, needs AQE)

Same co-location, but AQE right-sizes the partitions and splits any
hot key across tasks instead of OOMing one. Look for
`RebalancePartitions` in the plan.

In [ ]:
spark.conf.set("spark.sql.adaptive.enabled", "true")

orders.hint("rebalance", "country").explain()

(orders.hint("rebalance", "country")
 .write.mode("overwrite").partitionBy("country").parquet("output/rebalanced"))
print("rebalance hint:")
file_report("output/rebalanced")

spark.conf.set("spark.sql.adaptive.enabled", "false")

In [ ]:
import shutil
shutil.rmtree("output", ignore_errors=True)   # self-clean

## Exercises

1. Predict the partition histogram of
   `orders.repartition(2, "status")` before running it (orders has
   3 statuses). Which statuses share a partition? Verify with
   `spark_partition_id()`.
2. `sortWithinPartitions("country")` vs `orderBy("country")`: run
   `explain()` on both. Which nodes differ, and what does
   `global=false` on a Sort node mean physically?
3. Rerun the small-files demo writing the 20M-row `big` DataFrame
   from lesson 10.0 partitioned by `key % 10`. How bad is naive?
   Then compare `repartition(col)` vs the salted
   `repartition(col, (rand()*4).cast("int"))` variant on file
   count AND file sizes.
4. With AQE on, `repartition("country")` (no n) vs
   `repartition(8, "country")`: check the post-shuffle partition
   count of each. Which one did AQE touch, and why?
5. Build a case where `repartitionByRange(4, "order_date")` writes
   parquet files whose min/max date ranges don't overlap (check
   with `parquet-tools` or by reading each file individually), then
   show a date filter skipping files in the Spark UI's scan metrics.

In [ ]:
# your exercise code here